In [25]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import math

num_train = 2000

def feature_map(x):
    """
    Input:  [1, 28, 28]
    Output: [784, 2]
    """
    x = torch.stack(
        [
            torch.cos(math.pi * x / 2),
            torch.sin(math.pi * x / 2),
        ],
        dim=-1,
    )

    x = x.flatten(start_dim=0, end_dim=2)


    return x

# ([1,28,28],label)
image_size = 14
full_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            feature_map,
        ]
    ),
)
    


# select subset of MNIST
indices = torch.randperm(len(full_dataset))[:num_train]
train_dataset = Subset(full_dataset, indices)


#[64,1,28,28]
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
)

bond_dim = 20
input_dim = 2
num_sites = image_size**2

images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)



#test

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            feature_map,
        ]
    ),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
)

torch.Size([64, 196, 2])
torch.Size([64])


In [26]:
import torch
import torch.nn as nn


class MPS(nn.Module):
    def __init__(self, num_sites, physical_dim=2, bond_dim=3):
        super().__init__()

        if num_sites < 2:
            raise ValueError("num_sites must be at least 2")

        self.num_sites = num_sites
        self.physical_dim = physical_dim
        self.bond_dim = bond_dim

        tensors = []

        std = bond_dim ** (-(num_sites - 1) / (2 * num_sites))

        # First site: no left bond
        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim) * std
            )
        )

        # Middle sites: left and right bonds
        for _ in range(num_sites - 2):
            tensors.append(
                nn.Parameter(
                    torch.randn(physical_dim, bond_dim, bond_dim) * std
                )
            )

        # Final site: no right bond
        num_classes = 10

        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim, num_classes) * std
            )
        )

        self.tensors = nn.ParameterList(tensors)

    def forward(self, x):
        """
        x shape: [batch_size, num_sites, physical_dim]
        """
        
        if x.ndim != 3:
            raise ValueError(
                "x must have shape [batch_size, num_sites, physical_dim]"
            )

        if x.shape[1] != self.num_sites:
            raise ValueError(
                f"Expected {self.num_sites} sites, got {x.shape[1]}"
            )

        if x.shape[2] != self.physical_dim:
            raise ValueError(
                f"Expected physical_dim={self.physical_dim}, got {x.shape[2]}"
            )

        # First site:
        # [batch, physical_dim] × [physical_dim, bond_dim]
        # → [batch, bond_dim]
        # input of first site for each batch x 
        state = x[:, 0] @ self.tensors[0]

        # Middle sites:
        # Contract the physical input and the incoming bond.
        
        # indices:
        # b - runs through Batch
        # k - runs through inputs
        # d - tensor 
        for site in range(1, self.num_sites - 1):
            state = torch.einsum(
                "bl,bp,plr->br",
                state,
                x[:, site],
                self.tensors[site]
            )

        # Final site:
        # [batch, bond_dim] contracted with the final physical input
        # and the final MPS tensor → [batch]
        y = torch.einsum(
            "bl,bp,plc->bc",
            state,
            x[:, -1],
            self.tensors[-1]
        )

        return y



class MPS1(nn.Module):
    def __init__(
        self,
        num_sites,
        physical_dim=2,
        bond_dim=3,
        num_classes=10,
    ):
        super().__init__()

        if num_sites < 2:
            raise ValueError("num_sites must be at least 2")

        self.num_sites = num_sites
        self.physical_dim = physical_dim
        self.bond_dim = bond_dim
        self.num_classes = num_classes

        self.register_buffer(
            "identity",
            torch.eye(bond_dim),
        )

        tensors = []

        
        # First site
        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim) * 0.1
            )
        )

        # Middle sites: small perturbations around identity
        for _ in range(num_sites - 2):
            tensors.append(
                nn.Parameter(
                    torch.randn(
                        physical_dim,
                        bond_dim,
                        bond_dim,
                    ) * 1e-3
                )
            )

        # Final classification site
        tensors.append(
            nn.Parameter(
                torch.randn(
                    physical_dim,
                    bond_dim,
                    num_classes,
                ) * 0.1
            )
        )

        self.tensors = nn.ParameterList(tensors)

    def forward(self, x):
        """
        x shape: [batch_size, num_sites, physical_dim]
        returns: [batch_size, num_classes]
        """

        if x.ndim != 3:
            raise ValueError(
                "x must have shape [batch_size, num_sites, physical_dim]"
            )

        if x.shape[1] != self.num_sites:
            raise ValueError(
                f"Expected {self.num_sites} sites, got {x.shape[1]}"
            )

        if x.shape[2] != self.physical_dim:
            raise ValueError(
                f"Expected physical_dim={self.physical_dim}, got {x.shape[2]}"
            )

        # First site
        state = x[:, 0] @ self.tensors[0]

        # Middle sites
        for site in range(1, self.num_sites - 1):
            transition = torch.einsum(
                "bp,plr->blr",
                x[:, site],
                self.tensors[site],
            )

            transition = transition + self.identity

            state = torch.einsum(
                "bl,blr->br",
                state,
                transition,
            )

        # Final classification site
        logits = torch.einsum(
            "bl,bp,plc->bc",
            state,
            x[:, -1],
            self.tensors[-1],
        )

        return logits


In [ ]:
model = MPS1(num_sites=num_sites,physical_dim=input_dim, bond_dim=bond_dim)
print(model.bond_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(100):
    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_samples = 0

    for x, labels in train_loader:
        pred = model(x)
        loss = loss_fn(pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)

        train_loss_sum += loss.item() * batch_size
        train_correct += (
            pred.argmax(dim=1) == labels
        ).sum().item()

        train_samples += batch_size

    train_loss = train_loss_sum / train_samples
    train_accuracy = train_correct / train_samples

    # -------------------------
    # Testing
    # -------------------------
    model.eval()

    test_loss_sum = 0.0
    test_correct = 0
    test_samples = 0

    with torch.no_grad():
        for x, labels in test_loader:
            pred = model(x)
            loss = loss_fn(pred, labels)

            batch_size = x.size(0)

            test_loss_sum += loss.item() * batch_size
            test_correct += (
                pred.argmax(dim=1) == labels
            ).sum().item()

            test_samples += batch_size

    test_loss = test_loss_sum / test_samples
    test_accuracy = test_correct / test_samples

    print(
        f"EPOCH {epoch + 1:3d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_accuracy:.2%} | "
        f"test loss: {test_loss:.4f} | "
        f"test acc: {test_accuracy:.2%}"
    )

20
EPOCH   1 | train loss: 1.9675 | train acc: 33.95% | test loss: 1.9869 | test acc: 40.16%
EPOCH   2 | train loss: 0.9338 | train acc: 68.45% | test loss: 0.9083 | test acc: 67.61%
EPOCH   3 | train loss: 0.5043 | train acc: 83.40% | test loss: 0.6069 | test acc: 81.29%
EPOCH   4 | train loss: 0.4152 | train acc: 86.30% | test loss: 0.4068 | test acc: 87.99%
EPOCH   5 | train loss: 0.3133 | train acc: 89.85% | test loss: 0.5045 | test acc: 86.68%
EPOCH   6 | train loss: 0.2763 | train acc: 91.10% | test loss: 0.4479 | test acc: 86.68%
EPOCH   7 | train loss: 0.2371 | train acc: 91.60% | test loss: 0.4014 | test acc: 89.00%
EPOCH   8 | train loss: 0.2487 | train acc: 92.60% | test loss: 0.4081 | test acc: 88.58%
EPOCH   9 | train loss: 0.2374 | train acc: 91.95% | test loss: 0.4963 | test acc: 84.16%
EPOCH  10 | train loss: 0.3249 | train acc: 90.25% | test loss: 0.4314 | test acc: 88.01%
EPOCH  11 | train loss: 0.1900 | train acc: 94.40% | test loss: 0.3948 | test acc: 88.74%
EPOCH  